# EduLab AI — entraînement complet du professeur avec QLoRA

Ce notebook entraîne un adaptateur pédagogique sur **Qwen2.5-1.5B-Instruct**. Il est prévu pour un GPU Colab T4 (15 Go).

Avant de commencer : **Exécution → Modifier le type d’exécution → T4 GPU**. L’entraînement améliore le format et le comportement pédagogique, mais ne remplace ni le RAG, ni les sources officielles, ni une validation humaine.

In [ ]:
!nvidia-smi
!pip -q install -U 'transformers==4.53.3' 'datasets==4.0.0' 'peft==0.16.0' 'accelerate==1.9.0' 'bitsandbytes>=0.46.0' 'safetensors>=0.5.3' 'sentencepiece' 'matplotlib>=3.9'

## 1. Importer les trois jeux de données
Dans votre projet, ouvrez `data/processed/`, puis importez les trois fichiers demandés par la cellule suivante : train, validation et test.

In [ ]:
from google.colab import files
from pathlib import Path
import json, os, random, math, shutil, torch, pandas as pd

SEED = 20260728
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = Path('/content/edulab-teacher-qwen-1.5b-lora')
random.seed(SEED); torch.manual_seed(SEED)
uploaded = files.upload()
required = ['edulab_teacher_train.jsonl', 'edulab_teacher_validation.jsonl', 'edulab_teacher_test.jsonl']
missing = [name for name in required if name not in uploaded]
assert not missing, f'Fichiers manquants : {missing}'
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ABSENT')
assert torch.cuda.is_available(), 'Activez le GPU T4 dans les paramètres Colab.'

In [ ]:
def read_jsonl(name):
    return [json.loads(line) for line in Path('/content', name).read_text(encoding='utf-8').splitlines() if line.strip()]

splits = {
    'train': read_jsonl(required[0]),
    'validation': read_jsonl(required[1]),
    'test': read_jsonl(required[2]),
}
mandatory = {'instruction', 'context', 'response', 'class_name', 'subject', 'task'}
for split, rows in splits.items():
    assert rows, f'{split} est vide'
    assert all(mandatory.issubset(row) for row in rows), f'Schéma invalide dans {split}'
    assert all(len(row['response'].strip()) >= 20 for row in rows), f'Réponse trop courte dans {split}'
    print(split, len(rows), 'lignes')

assert not ({row.get('id') for row in splits['train']} & {row.get('id') for row in splits['test']}), 'Fuite train/test détectée'
frame = pd.DataFrame(splits['train'])
display(pd.crosstab(frame['class_name'], frame['subject']))
display(frame['task'].value_counts().rename('nombre').to_frame())
print('Statuts de validation :', frame.get('validation_status', pd.Series(['inconnu'])).value_counts().to_dict())

### Contrôle indispensable
Le corpus actuel contient surtout des exemples synthétiques ancrés dans le programme. Examinez manuellement un échantillon avant l’entraînement. Supprimez toute réponse scientifiquement fausse.

In [ ]:
sample = frame.sample(min(12, len(frame)), random_state=SEED)[['class_name','subject','task','instruction','response']]
pd.set_option('display.max_colwidth', 220)
display(sample)
CONFIRM_DATA_REVIEW = False  # Passez à True seulement après lecture de l'échantillon
assert CONFIRM_DATA_REVIEW, 'Lisez les exemples ci-dessus, puis passez CONFIRM_DATA_REVIEW à True.'

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
MAX_LENGTH = 768
SYSTEM = ('Tu es EduLab, un professeur pédagogique francophone. Réponds uniquement à partir du contexte fourni. '
          'Explique progressivement, vérifie les unités et les calculs, et signale toute information insuffisante.')

def encode(row):
    user = f"Niveau: {row['class_name']}\nMatière: {row['subject']}\nTâche: {row['task']}\nConsigne: {row['instruction']}\nContexte validé:\n{row['context']}"
    prompt_messages = [{'role':'system','content':SYSTEM}, {'role':'user','content':user}]
    full_messages = prompt_messages + [{'role':'assistant','content':row['response']}]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    encoded = tokenizer(full, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)['input_ids']
    labels = encoded['input_ids'].copy()
    labels[:min(len(prompt_ids), len(labels))] = [-100] * min(len(prompt_ids), len(labels))
    encoded['labels'] = labels
    return encoded

datasets = {}
for name, rows in splits.items():
    datasets[name] = Dataset.from_list(rows).map(encode, remove_columns=list(rows[0].keys()))
    kept = sum(any(label != -100 for label in item['labels']) for item in datasets[name])
    assert kept == len(datasets[name]), f'Réponses tronquées dans {name}'
    print(name, len(datasets[name]))

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant_config, device_map='auto', torch_dtype=compute_dtype)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, EarlyStoppingCallback

steps_per_epoch = max(1, math.ceil(len(datasets['train']) / (2 * 8)))
eval_steps = max(10, steps_per_epoch // 2)
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'), num_train_epochs=4, learning_rate=1e-4,
    per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=8,
    warmup_ratio=0.08, weight_decay=0.01, lr_scheduler_type='cosine',
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    eval_strategy='steps', save_strategy='steps', eval_steps=eval_steps, save_steps=eval_steps,
    logging_steps=5, save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, gradient_checkpointing=True, optim='paged_adamw_8bit',
    max_grad_norm=0.3, seed=SEED, data_seed=SEED, report_to='none',
)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)
trainer = Trainer(model=model, args=args, train_dataset=datasets['train'], eval_dataset=datasets['validation'], data_collator=collator, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
train_result = trainer.train()
validation_metrics = trainer.evaluate()
print(validation_metrics)

In [ ]:
import matplotlib.pyplot as plt
history = pd.DataFrame(trainer.state.log_history)
display(history.tail(10))
ax = history.dropna(subset=['loss']).plot(x='step', y='loss', label='train loss', figsize=(9,4))
if 'eval_loss' in history: history.dropna(subset=['eval_loss']).plot(x='step', y='eval_loss', label='validation loss', ax=ax)
plt.title('Courbes d’entraînement EduLab'); plt.grid(alpha=.2); plt.show()
test_metrics = trainer.evaluate(datasets['test'], metric_key_prefix='test')
if 'test_loss' in test_metrics: test_metrics['test_perplexity'] = float(math.exp(min(test_metrics['test_loss'], 20)))
print(test_metrics)

In [ ]:
# Test qualitatif avant export
model.eval(); model.config.use_cache = True
def generate(question, context, level='Troisième', subject='Physique-Chimie'):
    user = f'Niveau: {level}\nMatière: {subject}\nConsigne: {question}\nContexte validé:\n{context}'
    text = tokenizer.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':user}], tokenize=False, add_generation_prompt=True)
    batch = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad(): ids = model.generate(**batch, max_new_tokens=220, do_sample=False, repetition_penalty=1.08)
    return tokenizer.decode(ids[0][batch.input_ids.shape[1]:], skip_special_tokens=True)

checks = [
 ('Explique la loi d’Ohm et donne un exemple.', 'La loi d’Ohm relie U, R et I : U = R × I. U en volt, R en ohm et I en ampère.'),
 ('Que peux-tu conclure si le contexte ne contient pas la réponse ?', 'Le contexte ne fournit aucune information sur la photosynthèse.'),
]
for question, context in checks:
    print('QUESTION:', question, '\nRÉPONSE:', generate(question, context), '\n' + '-'*80)

In [ ]:
# Sauvegarde de l'adaptateur uniquement (pas de fusion : plus léger et réversible)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR / 'tokenizer')
metadata = {
 'base_model': MODEL_ID, 'method': 'QLoRA NF4', 'seed': SEED, 'max_length': MAX_LENGTH,
 'train_examples': len(datasets['train']), 'validation_examples': len(datasets['validation']), 'test_examples': len(datasets['test']),
 'train_metrics': train_result.metrics, 'validation_metrics': validation_metrics, 'test_metrics': test_metrics,
 'limitations': 'Corpus de petite taille; validation scientifique humaine et RAG obligatoires.'
}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
(OUTPUT_DIR / 'README.md').write_text(f'# EduLab Teacher QLoRA\n\nBase: `{MODEL_ID}`. Adaptateur pédagogique expérimental; ne pas utiliser sans RAG et validation humaine.\n', encoding='utf-8')
archive = shutil.make_archive('/content/edulab-teacher-qwen-1.5b-lora', 'zip', OUTPUT_DIR)
print('Archive créée :', archive, Path(archive).stat().st_size / 1e6, 'Mo')
files.download(archive)

## 2. Après le téléchargement

1. Décompressez l’archive dans `models/edulab-teacher-qwen-1.5b-lora/`.
2. Configurez `TEACHER_BASE_MODEL=Qwen/Qwen2.5-1.5B-Instruct`.
3. Configurez `TEACHER_ADAPTER_PATH` vers ce nouveau dossier.
4. Redémarrez le service `teacher-model`.
5. Lancez les évaluations avant de remplacer l’adaptateur actuel.

Le modèle 1.5B est plus lourd que le 0.5B : sur CPU local, les réponses resteront lentes. Pour une démonstration fluide, utilisez une instance GPU ou conservez OpenAI comme moteur final et le modèle local comme solution expérimentale.